# Trading Agent Performance

This notebook scores persisted Trading Agent candidates against the available `price_history` artifact.

Current read from the cached/latest artifacts after refresh:

- 34 generated candidates found.
- 20 candidates have at least one post-pick price bar and can be marked to market.
- 2 candidates have reached their stated horizon.
- Matured win rate: **50.0%**.
- Mark-to-market win rate on scored candidates: **45.0%**.
- Average signed return on scored candidates: **-2.52%**.
- Median signed return on scored candidates: **-2.76%**.

Interpretation guardrail: this is a tiny, mostly immature sample. Treat the matured win rate as the honest realized number, and the mark-to-market table as an early read only.

## Scoring Rules

- Each Trading Agent candidate is one pick. Repeated tickers across horizons are scored separately.
- `long` and `watch` win when price rises.
- `short` and `avoid` win when price falls.
- Entry price is the first daily close whose bar date is on or after the candidate `asof_time_utc` date.
- Target exit is the first close at or after the horizon date: 1w=7 days, 1m=30 days, 3m=91 days, 1y=365 days, 5y=1825 days.
- If the horizon has not arrived but a later close exists, the row is `mark_to_market`.
- If no later close exists, the row is `no_future_bar` and is excluded from win-rate calculations.
- No fees, borrow costs, slippage, sizing, or portfolio overlap adjustment are included.

## Current Snapshot Results

The current notebook run uses Trading Agent candidate snapshots cached/refreshed through June 23, 2026 and price history through the June 23, 2026 daily bar.

| Metric | Value |
|---|---:|
| Generated candidates | 34 |
| Candidates with post-pick price bars | 20 |
| Candidates at stated horizon | 2 |
| Candidates with no future bar yet | 14 |
| Matured win rate | 50.0% |
| Mark-to-market win rate on scored rows | 45.0% |
| Average signed return on scored rows | -2.52% |
| Median signed return on scored rows | -2.76% |

By horizon, using scored rows only for returns and win rate:

| Horizon | Total Candidates | Scored | Matured | Win Rate | Avg Signed Return | Median Signed Return |
|---|---:|---:|---:|---:|---:|---:|
| 1w | 11 | 7 | 2 | 71.4% | 0.63% | 3.41% |
| 1m | 8 | 4 | 0 | 50.0% | -0.48% | 1.05% |
| 3m | 7 | 4 | 0 | 25.0% | -5.31% | -8.89% |
| 1y | 4 | 2 | 0 | 0.0% | -9.32% | -9.32% |
| 5y | 4 | 3 | 0 | 33.3% | -4.34% | -2.76% |

By direction, using scored rows only for returns and win rate:

| Direction | Total Candidates | Scored | Matured | Win Rate | Avg Signed Return | Median Signed Return |
|---|---:|---:|---:|---:|---:|---:|
| avoid | 10 | 7 | 1 | 85.7% | 3.62% | 7.03% |
| long | 1 | 1 | 0 | 0.0% | -10.81% | -10.81% |
| watch | 23 | 12 | 1 | 25.0% | -5.41% | -7.28% |

Best scored rows: `ON` 1w avoid +11.05%, `MXL` 1w/3m avoid +7.58%, `GNRC` 1w/1m avoid +7.03%, `OSCR` 1m watch +4.86%.

Worst scored rows: `POET` 1w avoid -15.47% at maturity, `ON` watch rows -11.05%, `TSEM` long/watch rows -10.81%, `MXL` 1y watch -7.58%.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

cwd = Path.cwd().resolve()
if (cwd / "streamlit_alpaca_app").exists():
    APP_ROOT = cwd / "streamlit_alpaca_app"
elif cwd.name == "streamlit_alpaca_app":
    APP_ROOT = cwd
else:
    APP_ROOT = cwd.parent / "streamlit_alpaca_app"

if str(APP_ROOT) not in sys.path:
    sys.path.insert(0, str(APP_ROOT))

CACHE_ROOT = APP_ROOT / "cache" / "pipeline_store"
print(f"APP_ROOT={APP_ROOT}")
print(f"CACHE_ROOT={CACHE_ROOT}")

In [ ]:
def _cached_frame_versions(dataset_name: str) -> list[tuple[pd.DataFrame, str]]:
    out: list[tuple[pd.DataFrame, str]] = []
    for path in sorted((CACHE_ROOT / dataset_name).glob("*/frame.pkl")):
        frame = pd.read_pickle(path)
        if isinstance(frame, pd.DataFrame):
            out.append((frame.copy(), path.parent.name))
    return out


def _latest_remote_frame(dataset_name: str) -> tuple[pd.DataFrame | None, str]:
    try:
        from services.pipeline_store import load_latest_dataset_frame

        frame, meta = load_latest_dataset_frame(dataset_name)
        version = getattr(meta, "dataset_version_id", "remote_latest") if meta is not None else "remote_latest"
        if isinstance(frame, pd.DataFrame):
            return frame.copy(), str(version)
    except Exception as exc:
        print(f"Remote latest load failed for {dataset_name}: {type(exc).__name__}: {exc}")
    return None, ""


def load_dataset_versions(dataset_name: str, *, include_remote_latest: bool = True) -> pd.DataFrame:
    parts: list[pd.DataFrame] = []
    for frame, version in _cached_frame_versions(dataset_name):
        chunk = frame.copy()
        chunk["_dataset_version_id"] = version
        chunk["_dataset_source"] = "cache"
        parts.append(chunk)

    if include_remote_latest:
        frame, version = _latest_remote_frame(dataset_name)
        if frame is not None:
            chunk = frame.copy()
            chunk["_dataset_version_id"] = version
            chunk["_dataset_source"] = "remote_latest"
            parts.append(chunk)

    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()


runs_raw = load_dataset_versions("trading_agent_runs")
candidates_raw = load_dataset_versions("trading_agent_candidates")
price_raw = load_dataset_versions("price_history")

print({
    "runs_raw": len(runs_raw),
    "candidates_raw": len(candidates_raw),
    "price_raw": len(price_raw),
})

In [ ]:
def clean_text(value: object) -> str:
    if value is None:
        return ""
    text = str(value).strip()
    return "" if text.lower() == "nan" else text


runs = runs_raw.copy()
if not runs.empty and "trading_agent_run_id" in runs.columns:
    runs = runs.drop_duplicates(subset=["trading_agent_run_id"], keep="last").reset_index(drop=True)

candidates = candidates_raw.copy()
if not candidates.empty:
    for col in ["ticker", "direction", "horizon_key", "candidate_id", "run_id"]:
        if col in candidates.columns:
            candidates[col] = candidates[col].map(clean_text)
    if "ticker" in candidates.columns:
        candidates["ticker"] = candidates["ticker"].str.upper()
    if "candidate_id" in candidates.columns:
        candidates = candidates.drop_duplicates(subset=["candidate_id"], keep="last").reset_index(drop=True)

prices = price_raw.copy()
if not prices.empty:
    prices["symbol"] = prices["symbol"].map(clean_text).str.upper()
    prices["timestamp"] = pd.to_datetime(prices["timestamp"], utc=True, errors="coerce")
    prices["close"] = pd.to_numeric(prices["close"], errors="coerce")
    prices = (
        prices.dropna(subset=["symbol", "timestamp", "close"])
        .sort_values(["symbol", "timestamp"])
        .drop_duplicates(subset=["symbol", "timestamp"], keep="last")
        .reset_index(drop=True)
    )

print("Trading Agent runs:", len(runs))
print("Trading Agent candidates:", len(candidates))
print("Price rows:", len(prices))
print("Price date range:", prices["timestamp"].min(), "->", prices["timestamp"].max())
print("Price symbols:", prices["symbol"].nunique())

runs[[c for c in ["run_id", "horizon_key", "status", "candidate_count", "asof_time_utc", "generated_at_utc", "_dataset_version_id"] if c in runs.columns]].sort_values(["asof_time_utc", "horizon_key"]).tail(20)

In [ ]:
HORIZON_DAYS = {
    "1w": 7,
    "1m": 30,
    "3m": 91,
    "1y": 365,
    "5y": 365 * 5,
}


def direction_multiplier(direction: object) -> int:
    text = clean_text(direction).lower()
    return -1 if text in {"short", "avoid"} else 1


def score_candidate(row: pd.Series, price_frame: pd.DataFrame) -> dict[str, object]:
    ticker = clean_text(row.get("ticker")).upper()
    asof = pd.to_datetime(row.get("asof_time_utc") or row.get("generated_at_utc"), utc=True, errors="coerce")
    out = row.to_dict()
    out["asof_ts"] = asof

    symbol_prices = price_frame[price_frame["symbol"].eq(ticker)].copy()
    if symbol_prices.empty:
        out["outcome_status"] = "missing_price"
        return out
    if pd.isna(asof):
        out["outcome_status"] = "missing_asof"
        return out

    symbol_prices["bar_date"] = symbol_prices["timestamp"].dt.date
    entry_candidates = symbol_prices[symbol_prices["bar_date"] >= asof.date()]
    if entry_candidates.empty:
        out["outcome_status"] = "missing_entry"
        return out

    entry = entry_candidates.iloc[0]
    future = symbol_prices[symbol_prices["timestamp"] > entry["timestamp"]]
    out["entry_ts"] = entry["timestamp"]
    out["entry_close"] = float(entry["close"])

    if future.empty:
        out["outcome_status"] = "no_future_bar"
        return out

    horizon_key = clean_text(row.get("horizon_key"))
    target_date = asof + pd.Timedelta(days=HORIZON_DAYS.get(horizon_key, 0))
    target_candidates = future[future["timestamp"] >= target_date]
    is_matured = not target_candidates.empty
    exit_row = target_candidates.iloc[0] if is_matured else future.iloc[-1]

    raw_return = (float(exit_row["close"]) / float(entry["close"]) - 1.0) * 100.0
    signed_return = raw_return * direction_multiplier(row.get("direction"))

    out.update(
        {
            "target_date": target_date,
            "exit_ts": exit_row["timestamp"],
            "exit_close": float(exit_row["close"]),
            "raw_return_pct": raw_return,
            "signed_return_pct": signed_return,
            "is_matured": bool(is_matured),
            "is_win": bool(signed_return > 0),
            "outcome_status": "matured" if is_matured else "mark_to_market",
        }
    )
    return out


performance = pd.DataFrame([score_candidate(row, prices) for _, row in candidates.iterrows()])
performance["generated_at_utc"] = pd.to_datetime(performance.get("generated_at_utc"), utc=True, errors="coerce")
performance["asof_time_utc"] = pd.to_datetime(performance.get("asof_time_utc"), utc=True, errors="coerce")

cols = [
    "asof_time_utc", "horizon_key", "ticker", "direction", "rank", "outcome_status",
    "entry_ts", "entry_close", "exit_ts", "exit_close", "signed_return_pct", "is_win",
    "setup", "hypothesis",
]
performance[[c for c in cols if c in performance.columns]].sort_values(["asof_time_utc", "horizon_key", "rank"]).reset_index(drop=True)

In [ ]:
def summarize(frame: pd.DataFrame) -> pd.Series:
    scored = frame[frame["signed_return_pct"].notna()] if "signed_return_pct" in frame.columns else frame.iloc[0:0]
    matured = frame[frame["outcome_status"].eq("matured")] if "outcome_status" in frame.columns else frame.iloc[0:0]
    return pd.Series(
        {
            "candidates": len(frame),
            "scored": len(scored),
            "matured": len(matured),
            "no_future_bar": int(frame["outcome_status"].eq("no_future_bar").sum()) if "outcome_status" in frame.columns else 0,
            "missing_price_or_entry": int(frame["outcome_status"].isin(["missing_price", "missing_entry", "missing_asof"]).sum()) if "outcome_status" in frame.columns else 0,
            "win_rate_scored": scored["is_win"].mean() if len(scored) else np.nan,
            "win_rate_matured": matured["is_win"].mean() if len(matured) else np.nan,
            "avg_signed_return_scored": scored["signed_return_pct"].mean() if len(scored) else np.nan,
            "median_signed_return_scored": scored["signed_return_pct"].median() if len(scored) else np.nan,
        }
    )


overall_summary = summarize(performance).to_frame("value")
by_horizon = performance.groupby("horizon_key", dropna=False).apply(summarize, include_groups=False).reset_index()
by_direction = performance.groupby("direction", dropna=False).apply(summarize, include_groups=False).reset_index()
by_snapshot = performance.groupby("_dataset_version_id", dropna=False).apply(summarize, include_groups=False).reset_index()

print("Overall")
display(overall_summary)

print("By horizon")
display(by_horizon.sort_values("horizon_key"))

print("By direction")
display(by_direction.sort_values("direction"))

print("Outcome status counts")
display(performance["outcome_status"].value_counts(dropna=False).to_frame("count"))

In [ ]:
scored = performance[performance["signed_return_pct"].notna()].copy()

best = scored.sort_values("signed_return_pct", ascending=False).head(10)
worst = scored.sort_values("signed_return_pct", ascending=True).head(10)

print("Best signed returns")
display(best[["ticker", "horizon_key", "direction", "outcome_status", "signed_return_pct", "entry_ts", "exit_ts", "setup"]])

print("Worst signed returns")
display(worst[["ticker", "horizon_key", "direction", "outcome_status", "signed_return_pct", "entry_ts", "exit_ts", "setup"]])

In [ ]:
if not scored.empty:
    fig = px.histogram(
        scored,
        x="signed_return_pct",
        color="direction",
        nbins=24,
        title="Signed Return Distribution: Scored Candidates",
        labels={"signed_return_pct": "Signed return (%)"},
    )
    fig.show()

    horizon_plot = by_horizon.copy()
    horizon_plot["win_rate_scored_pct"] = horizon_plot["win_rate_scored"] * 100
    fig = px.bar(
        horizon_plot,
        x="horizon_key",
        y="win_rate_scored_pct",
        text="scored",
        title="Mark-to-Market Win Rate by Horizon",
        labels={"win_rate_scored_pct": "Win rate (%)", "horizon_key": "Horizon"},
    )
    fig.update_yaxes(range=[0, 100])
    fig.show()

    fig = px.scatter(
        scored,
        x="entry_ts",
        y="signed_return_pct",
        color="direction",
        symbol="outcome_status",
        hover_data=["ticker", "horizon_key", "setup"],
        title="Candidate Returns by Entry Date",
        labels={"entry_ts": "Entry close", "signed_return_pct": "Signed return (%)"},
    )
    fig.show()

In [ ]:
try:
    from services.pipeline_store import trading_agent_actions_table

    actions = trading_agent_actions_table(limit=500)
except Exception as exc:
    print(f"Could not load action log: {type(exc).__name__}: {exc}")
    actions = pd.DataFrame()

if actions.empty:
    print("No Place / Reject action rows were available from Postgres in this environment.")
else:
    display(actions)

In [ ]:
# Optional export for deeper analysis outside the notebook.
export_dir = APP_ROOT / "documents" / "debug"
export_dir.mkdir(parents=True, exist_ok=True)
export_path = export_dir / "trading_agent_performance_scored.csv"
performance.to_csv(export_path, index=False)
print(export_path)